# Chapter 4: Expiration profit and loss in code

Describe a position as a list of legs, then get its value, P&L, slopes and breakevens at expiration.
Notes: [[chapters/04-expiration-profit-and-loss]]

A leg is `(quantity, kind, strike, price)`. `kind` is `"call"`, `"put"` or `"und"` (underlying; strike is `None` and price is the entry price). Quantity is positive for long and negative for short.

In [1]:
def leg_value(kind, strike, S):
    """Value of ONE contract at expiration."""
    if kind == "call":
        return max(0, S - strike)
    if kind == "put":
        return max(0, strike - S)
    return S                                   # underlying

def position_value(legs, S):
    """Parity-graph value of the position (ignores prices paid)."""
    return sum(q * leg_value(k, K, S) for q, k, K, p in legs)

def pnl(legs, S, offset=0.0):
    """Profit and loss at expiration. offset lets you shift the graph if you only know one P&L point."""
    return sum(q * (leg_value(k, K, S) - p) for q, k, K, p in legs) + offset

def strikes(legs):
    return sorted({K for q, k, K, p in legs if K is not None})

def slopes(legs):
    """Slope in each stretch: below the lowest strike, between strikes, above the highest."""
    ks = strikes(legs)
    pts = [ks[0] - 1] + [ (a + b) / 2 for a, b in zip(ks, ks[1:]) ] + [ks[-1] + 1]
    out = []
    for x in pts:                              # slope = change in value for a 1-point step
        out.append(position_value(legs, x + 0.5) - position_value(legs, x - 0.5))
    return list(zip(["below %g" % ks[0]] + ["%g-%g" % (a, b) for a, b in zip(ks, ks[1:])] + ["above %g" % ks[-1]], out))

def breakevens(legs, offset=0.0):
    """Underlying prices where P&L is zero, found stretch by stretch with x = x0 - y0/m."""
    ks = strikes(legs)
    sl = [m for _, m in slopes(legs)]
    found = []
    # tail below the lowest strike
    y0, m = pnl(legs, ks[0], offset), sl[0]
    if m != 0 and ks[0] - y0 / m < ks[0]:
        found.append(ks[0] - y0 / m)
    # between strikes
    for i in range(len(ks) - 1):
        a, b = ks[i], ks[i + 1]
        ya, yb, m = pnl(legs, a, offset), pnl(legs, b, offset), sl[i + 1]
        if ya == 0:
            found.append(a)
        elif m != 0 and ya * yb < 0:
            found.append(a - ya / m)
    # tail above the highest strike
    y0, m = pnl(legs, ks[-1], offset), sl[-1]
    if m != 0 and ks[-1] - y0 / m > ks[-1]:
        found.append(ks[-1] - y0 / m)
    return sorted(set(round(x, 6) for x in found))

## Single options: P&L and breakeven

In [2]:
long_call = [(1, "call", 100, 3.50)]
short_put = [(-1, "put", 95, 2.25)]
print("Long 100 call @3.50: P&L at 90, 100, 102, 103.5, 110 =", [pnl(long_call, s) for s in (90, 100, 102, 103.5, 110)])
print("  breakeven:", breakevens(long_call), "(= 100 + 3.50)")
print("Short 95 put @2.25: P&L at 90, 92.75, 95, 100 =", [pnl(short_put, s) for s in (90, 92.75, 95, 100)])
print("  breakeven:", breakevens(short_put), "(= 95 - 2.25)")

calls = {95: 6.25, 100: 3.50, 105: 1.75}
puts  = {95: 2.25, 100: 4.50, 105: 7.75}
print()
print("strike  call breakeven  put breakeven")
for X in (95, 100, 105):
    print(f"{X:>6} {breakevens([(1, 'call', X, calls[X])])[0]:>14.2f} {breakevens([(1, 'put', X, puts[X])])[0]:>14.2f}")

Long 100 call @3.50: P&L at 90, 100, 102, 103.5, 110 = [-3.5, -3.5, -1.5, 0.0, 6.5]
  breakeven: [103.5] (= 100 + 3.50)
Short 95 put @2.25: P&L at 90, 92.75, 95, 100 = [-2.75, 0.0, 2.25, 2.25]
  breakeven: [92.75] (= 95 - 2.25)

strike  call breakeven  put breakeven
    95         101.25          92.75
   100         103.50          95.50
   105         106.75          97.25


## Four-leg position (Example A)

In [3]:
posA = [(1, "call", 95, 6.25), (-1, "call", 105, 1.75), (-2, "put", 105, 7.75), (-2, "und", None, 98.00)]
print("slopes:", slopes(posA))
print("P&L at 95:", pnl(posA, 95), " at 105:", pnl(posA, 105), " at 110:", pnl(posA, 110))
print("breakevens:", breakevens(posA))

slopes: [('below 95', 0.0), ('95-105', 1.0), ('above 105', -2.0)]
P&L at 95: -3.0  at 105: 7.0  at 110: -3.0
breakevens: [98.0, 108.5]


## Eight-leg position (Example B)
The book only tells us the P&L at 62.00 (+2.10), so we compute the *shape* with zero prices and shift it with `offset`.

In [4]:
posB = [(-4, "und", None, 0), (3, "call", 65, 0), (2, "put", 65, 0), (2, "call", 70, 0), (-4, "put", 70, 0),
        (-6, "call", 75, 0), (3, "put", 75, 0), (4, "call", 80, 0), (-2, "put", 80, 0)]
print("slopes:", slopes(posB))
offset = 2.10 - pnl(posB, 62.00)               # make P&L(62) = +2.10
for s in (62, 65, 70, 75, 80, 81.5):
    print(f"P&L at {s:>5}: {pnl(posB, s, offset):+.2f}")
print("breakevens:", breakevens(posB, offset))

slopes: [('below 65', -3.0), ('65-70', 2.0), ('70-75', 0.0), ('75-80', -3.0), ('above 80', -1.0)]
P&L at    62: +2.10
P&L at    65: -6.90
P&L at    70: +3.10
P&L at    75: +3.10
P&L at    80: -11.90
P&L at  81.5: -13.40
breakevens: [62.7, 68.45, 76.033333]


## Plot a position

In [5]:
try:
    import numpy as np
    import matplotlib.pyplot as plt

    def plot_pnl(legs, lo, hi, offset=0.0, title=""):
        xs = np.linspace(lo, hi, 400)
        ys = [pnl(legs, x, offset) for x in xs]
        plt.figure(figsize=(8, 4.5))
        plt.plot(xs, ys, lw=2.5)
        plt.axhline(0, color="grey", lw=1)
        for b in breakevens(legs, offset):
            if lo <= b <= hi:
                plt.plot([b], [0], "o", color="crimson")
        for k in strikes(legs):
            plt.axvline(k, color="grey", ls=":", lw=1)
        plt.xlabel("Underlying price at expiration"); plt.ylabel("Profit or loss")
        plt.title(title); plt.show()

    plot_pnl(posA, 85, 115, title="Example A")
    plot_pnl(posB, 58, 86, offset, title="Example B")
except ImportError:
    print("matplotlib not installed; skipping plots")

## Try it
- Change a price in `posA` and see how the breakevens move.
- Add a leg and check the slopes table changes only at that leg's strike.
- Build "long call + short put at the same strike" and confirm the slope is +1 in every stretch.